# Agriculture & Climate SLM — Competition Pipeline v7

This notebook is a **local competition pipeline**:

**local venv → run notebook → generate `submission.csv` → upload CSV to Kaggle → compare leaderboard score**

The production algorithm is not a generic chatbot. The competition metric is character-level Levenshtein distance, so the system explicitly learns to choose the **known training reference answer most likely to minimize that metric**.

### What changed from v6

v6 used hand-selected global retrieval weights and then copied the top reference. That is not enough for this benchmark.

v7 uses:

1. **Document routing** — infer which of the 24 source documents the test question belongs to.
2. **Candidate expansion** — consider references from the best document routes plus strong global lexical matches.
3. **Metric-aware reranking** — train a small tabular model on training-only examples to predict the Levenshtein distance between a candidate reference and the hidden target answer.
4. **Strict outer-fold validation** — the reranker never sees the validation reference while selecting its candidate.
5. **Reference preservation** — candidate answers are copied exactly; no sentence truncation or rewriting.
6. **Optional local SLM hook** — deliberately not required for the default submission path.

No Hugging Face download, authentication, Kaggle GPU, or internet model access is required.


## 1. Load data

The notebook expects:

- `train_qa.csv`
- `test_questions.csv`
- `documents.csv`

Place them beside the notebook or set `DATA_DIR` explicitly.


In [1]:

from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings("ignore")

COMPETITION_SLUG = "agriculture-climate-slm"

def find_data_dir():
    candidates = [
        Path("."),
        Path("/home/ui_hamilton/Documents/personal/agriculture-climate-slm-challenge"),
    ]
    for base in candidates:
        if (base / "train_qa.csv").exists() and (base / "test_questions.csv").exists() and (base / "documents.csv").exists():
            return base
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("train_qa.csv"):
            if (p.parent / "test_questions.csv").exists() and (p.parent / "documents.csv").exists():
                return p.parent
    return Path(".")

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

docs = pd.read_csv(DATA_DIR / "documents.csv")
train = pd.read_csv(DATA_DIR / "train_qa.csv")
test = pd.read_csv(DATA_DIR / "test_questions.csv")

required_train = {"question", "topic", "crop", "agro_zone", "document_id", "reference_answer"}
required_test = {"QuestionId", "question", "topic", "crop", "agro_zone"}
required_docs = {"document_id", "text"}

missing = {
    "train": required_train - set(train.columns),
    "test": required_test - set(test.columns),
    "docs": required_docs - set(docs.columns),
}
if any(missing.values()):
    raise ValueError(f"Missing columns: {missing}")

train = train.reset_index(drop=True).copy()
test = test.reset_index(drop=True).copy()
docs = docs.reset_index(drop=True).copy()

print("DATA_DIR:", DATA_DIR.resolve())
print(f"documents={len(docs)}  train={len(train)}  test={len(test)}")


DATA_DIR: /home/ui_hamilton/Documents/personal/agriculture-climate-slm-challenge
documents=24  train=45  test=12


## 2. Text normalization and exact-reference handling

Reference answers are ground truth. We never clean, truncate, capitalize, paraphrase, or rewrite them before submission.

In [2]:

def norm(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text)).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s%/-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def reference(text):
    return "" if text is None else str(text).strip()

def lev_sim(a, b):
    a, b = str(a), str(b)
    if not a and not b:
        return 1.0
    return 1.0 - Levenshtein.distance(a, b) / max(len(a), len(b), 1)

def word_overlap(a, b):
    A, B = set(norm(a).split()), set(norm(b).split())
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def metadata_features(row_a, row_b):
    vals = []
    for col in ("topic", "crop", "agro_zone"):
        vals.append(float(norm(row_a.get(col, "")) == norm(row_b.get(col, ""))))
    return vals


## 3. Feature engine

The important change is that we do **not** decide that a TF-IDF score of 0.71 plus a fuzzy score of 0.83 means anything by hand.

Instead, every `(question, candidate reference)` pair becomes a feature vector. The reranker learns which combinations historically correspond to low target-answer Levenshtein distance.

Features include:

- word and character TF-IDF similarity
- several fuzzy/edit similarities
- exact metadata agreement
- question-to-document similarity
- candidate-answer lexical relationship to the question
- answer length characteristics
- document-route rank


In [3]:

class FeatureEngine:
    def __init__(self, corpus_df, docs_df):
        self.df = corpus_df.reset_index(drop=True).copy()
        self.docs = docs_df.reset_index(drop=True).copy()

        self.questions = self.df["question"].fillna("").map(norm).tolist()

        self.word_vec = TfidfVectorizer(
            ngram_range=(1, 3),
            sublinear_tf=True,
            min_df=1,
            max_df=1.0,
        )
        self.char_vec = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 6),
            sublinear_tf=True,
            min_df=1,
        )

        self.Xw = self.word_vec.fit_transform(self.questions)
        self.Xc = self.char_vec.fit_transform(self.questions)

        self.doc_ids = self.docs["document_id"].astype(str).tolist()
        self.doc_texts = self.docs["text"].fillna("").map(norm).tolist()

        self.doc_word_vec = TfidfVectorizer(
            ngram_range=(1, 3),
            sublinear_tf=True,
            min_df=1,
        )
        self.doc_char_vec = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 6),
            sublinear_tf=True,
            min_df=1,
        )
        self.Dw = self.doc_word_vec.fit_transform(self.doc_texts)
        self.Dc = self.doc_char_vec.fit_transform(self.doc_texts)
        self.doc_pos = {d: i for i, d in enumerate(self.doc_ids)}

    def question_similarity(self, query_text, idx):
        q = norm(query_text)
        qw = self.word_vec.transform([q])
        qc = self.char_vec.transform([q])

        word = float(cosine_similarity(qw, self.Xw[idx])[0, 0])
        char = float(cosine_similarity(qc, self.Xc[idx])[0, 0])
        cand = self.questions[idx]

        return {
            "q_word": word,
            "q_char": char,
            "q_ratio": fuzz.ratio(q, cand) / 100.0,
            "q_token_set": fuzz.token_set_ratio(q, cand) / 100.0,
            "q_token_sort": fuzz.token_sort_ratio(q, cand) / 100.0,
            "q_wratio": fuzz.WRatio(q, cand) / 100.0,
            "q_partial": fuzz.partial_ratio(q, cand) / 100.0,
            "q_lev": lev_sim(q, cand),
            "q_jw": JaroWinkler.normalized_similarity(q, cand),
        }

    def document_similarity(self, query_text, document_id):
        pos = self.doc_pos.get(str(document_id))
        if pos is None:
            return 0.0, 0.0

        q = norm(query_text)
        qw = self.doc_word_vec.transform([q])
        qc = self.doc_char_vec.transform([q])

        return (
            float(cosine_similarity(qw, self.Dw[pos])[0, 0]),
            float(cosine_similarity(qc, self.Dc[pos])[0, 0]),
        )

    def pair_features(self, query_row, candidate_idx, doc_rank=99, doc_score=0.0):
        cand = self.df.iloc[candidate_idx]
        qtext = query_row["question"]
        ans = reference(cand["reference_answer"])

        feats = self.question_similarity(qtext, candidate_idx)
        dw, dc = self.document_similarity(qtext, cand["document_id"])

        qmeta = metadata_features(query_row, cand)

        feats.update({
            "doc_word": dw,
            "doc_char": dc,
            "meta_topic": qmeta[0],
            "meta_crop": qmeta[1],
            "meta_zone": qmeta[2],
            "meta_sum": sum(qmeta),
            "route_rank": float(doc_rank),
            "route_score": float(doc_score),
            "answer_len": float(len(ans)),
            "question_len": float(len(str(qtext))),
            "answer_question_len_ratio": float(len(ans) / max(len(str(qtext)), 1)),
            "answer_q_word_overlap": word_overlap(ans, qtext),
            "answer_q_ratio": fuzz.ratio(ans, qtext) / 100.0,
            "answer_q_token_set": fuzz.token_set_ratio(ans, qtext) / 100.0,
        })
        return feats

    @property
    def feature_names(self):
        return [
            "q_word", "q_char", "q_ratio", "q_token_set", "q_token_sort",
            "q_wratio", "q_partial", "q_lev", "q_jw",
            "doc_word", "doc_char",
            "meta_topic", "meta_crop", "meta_zone", "meta_sum",
            "route_rank", "route_score",
            "answer_len", "question_len", "answer_question_len_ratio",
            "answer_q_word_overlap", "answer_q_ratio", "answer_q_token_set",
        ]


## 4. Document router

The training data tells us which document generated each answer. That signal should be used explicitly.

For each query we score all 24 documents using:

- query ↔ document TF-IDF
- metadata agreement
- similarity to training questions already assigned to that document

The top document routes form the candidate pool. We still retain strong global question matches so a routing mistake cannot completely eliminate a good reference.


In [4]:

class DocumentRouter:
    def __init__(self, corpus_df, docs_df):
        self.train = corpus_df.reset_index(drop=True)
        self.docs = docs_df.reset_index(drop=True)

        self.doc_ids = self.docs["document_id"].astype(str).tolist()
        self.doc_meta = {
            str(r["document_id"]): r for _, r in self.docs.iterrows()
        }

        self.doc_vec = TfidfVectorizer(
            ngram_range=(1, 3),
            sublinear_tf=True,
            min_df=1,
        )
        self.doc_char = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 6),
            sublinear_tf=True,
            min_df=1,
        )
        doc_text = (
            self.docs["title"].fillna("").astype(str) + " " +
            self.docs["text"].fillna("").astype(str)
            if "title" in self.docs.columns
            else self.docs["text"].fillna("").astype(str)
        )
        self.Dw = self.doc_vec.fit_transform(doc_text.map(norm))
        self.Dc = self.doc_char.fit_transform(doc_text.map(norm))

        self.doc_train_indices = {}
        for i, r in self.train.iterrows():
            self.doc_train_indices.setdefault(str(r["document_id"]), []).append(i)

        self.fe = FeatureEngine(self.train, self.docs)

    def score(self, query_row, document_id):
        did = str(document_id)
        pos = self.doc_ids.index(did)

        q = norm(query_row["question"])
        qw = self.doc_vec.transform([q])
        qc = self.doc_char.transform([q])

        dw = float(cosine_similarity(qw, self.Dw[pos])[0, 0])
        dc = float(cosine_similarity(qc, self.Dc[pos])[0, 0])

        meta = self.doc_meta[did]
        meta_vals = [
            float(norm(query_row.get("topic", "")) == norm(meta.get("topic", ""))) if "topic" in meta else 0.0,
            float(norm(query_row.get("crop", "")) == norm(meta.get("crop", ""))) if "crop" in meta else 0.0,
            float(norm(query_row.get("agro_zone", "")) == norm(meta.get("agro_zone", ""))) if "agro_zone" in meta else 0.0,
        ]

        train_idxs = self.doc_train_indices.get(did, [])
        if train_idxs:
            sims = []
            for idx in train_idxs:
                s = self.fe.question_similarity(q, idx)
                sims.append(0.55 * s["q_word"] + 0.45 * s["q_char"])
            train_support = max(sims)
        else:
            train_support = 0.0

        score = (
            0.35 * dw
            + 0.20 * dc
            + 0.15 * train_support
            + 0.20 * np.mean(meta_vals)
            + 0.10 * max(meta_vals)
        )
        return float(score)

    def route(self, query_row, top_n=4):
        scored = [
            (self.score(query_row, did), did)
            for did in self.doc_ids
        ]
        return sorted(scored, reverse=True)[:top_n]


## 5. Candidate generation

Candidate generation is deliberately generous. The reranker, not the retrieval heuristic, decides the final reference.

In [5]:

def candidate_indices(query_row, retriever, router, top_docs=4):
    # Strong document routes.
    routes = router.route(query_row, top_n=top_docs)

    candidate = set()

    for rank, (doc_score, did) in enumerate(routes, 1):
        for idx in router.doc_train_indices.get(str(did), []):
            candidate.add(idx)

    # Strong global question matches. This protects against document-routing errors.
    q = norm(query_row["question"])
    global_scores = []
    for idx, cand_q in enumerate(retriever.questions):
        global_scores.append((
            0.35 * fuzz.ratio(q, cand_q)
            + 0.30 * fuzz.token_set_ratio(q, cand_q)
            + 0.20 * fuzz.WRatio(q, cand_q)
            + 0.15 * (100.0 * lev_sim(q, cand_q)),
            idx
        ))

    for _, idx in sorted(global_scores, reverse=True)[:12]:
        candidate.add(idx)

    return sorted(candidate), routes


## 6. Metric-aware reranker

This is the key component.

For a training query, every candidate has a known target:

`Levenshtein(candidate_reference, true_reference)`

We train a small ExtraTrees regressor to predict that distance from retrieval features.

The model never receives the true reference as an input feature. During validation, the validation row is excluded from the candidate corpus and its reference is never used to train the reranker.

This is a much closer optimization target than a hand-written similarity score.


In [6]:

def build_pair_dataset(fit_df, candidate_engine, router, include_all=True):
    rows = []
    targets = []

    local = fit_df.reset_index(drop=True)

    # Rebuild the feature engine/router against this fold only.
    fe = FeatureEngine(local, docs)
    rt = DocumentRouter(local, docs)

    for qi, qrow in local.iterrows():
        # Candidate generation must exclude the query's own row.
        candidates, routes = candidate_indices(qrow, fe, rt, top_docs=4)
        route_map = {did: rank for rank, (_, did) in enumerate(routes, 1)}
        route_score_map = {did: score for score, did in routes}

        true_answer = reference(qrow["reference_answer"])

        for ci in candidates:
            if ci == qi:
                continue

            cand = local.iloc[ci]
            did = str(cand["document_id"])
            route_rank = route_map.get(did, 99)
            route_score = route_score_map.get(did, 0.0)

            f = fe.pair_features(
                qrow,
                ci,
                doc_rank=route_rank,
                doc_score=route_score,
            )
            rows.append([f[name] for name in fe.feature_names])

            # This target is exactly the competition's loss contribution
            # for choosing this candidate reference.
            targets.append(
                Levenshtein.distance(reference(cand["reference_answer"]), true_answer)
            )

    return np.asarray(rows, dtype=float), np.asarray(targets, dtype=float), fe.feature_names

def train_reranker(fit_df):
    X, y, feature_names = build_pair_dataset(fit_df, None, None)

    model = ExtraTreesRegressor(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        max_features=0.80,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X, y)
    return model, feature_names

def predict_with_reranker(query_row, fit_df, model):
    local = fit_df.reset_index(drop=True)
    fe = FeatureEngine(local, docs)
    rt = DocumentRouter(local, docs)

    candidates, routes = candidate_indices(query_row, fe, rt, top_docs=4)

    route_map = {did: rank for rank, (_, did) in enumerate(routes, 1)}
    route_score_map = {did: score for score, did in routes}

    feature_rows = []
    usable = []

    for ci in candidates:
        cand = local.iloc[ci]
        did = str(cand["document_id"])
        f = fe.pair_features(
            query_row,
            ci,
            doc_rank=route_map.get(did, 99),
            doc_score=route_score_map.get(did, 0.0),
        )
        feature_rows.append([f[name] for name in fe.feature_names])
        usable.append(ci)

    if not usable:
        # Should never happen with the supplied competition structure.
        idx = 0
        return reference(local.iloc[idx]["reference_answer"]), {
            "candidate_index": idx,
            "predicted_distance": float("inf"),
            "routes": routes,
        }

    predicted_distance = model.predict(np.asarray(feature_rows))
    order = np.argsort(predicted_distance)

    best_pos = int(order[0])
    best_idx = usable[best_pos]

    return reference(local.iloc[best_idx]["reference_answer"]), {
        "candidate_index": best_idx,
        "predicted_distance": float(predicted_distance[best_pos]),
        "routes": routes,
        "candidate_count": len(usable),
    }


## 7. Honest outer-fold validation

We evaluate the complete pipeline, not just a retrieval score.

For each outer fold:

- fit the candidate/reranker system only on the training portion;
- retrieve candidates for validation questions;
- predict the candidate whose **predicted target Levenshtein** is lowest;
- compare against the untouched validation reference.

We also report a simple lexical baseline so a more complicated system is not accepted merely because it looks sophisticated.


In [7]:

from sklearn.model_selection import KFold

def lexical_baseline_predict(query_row, fit_df):
    local = fit_df.reset_index(drop=True)
    q = norm(query_row["question"])

    scores = []
    for i, cand in local.iterrows():
        cq = norm(cand["question"])
        score = (
            0.25 * fuzz.ratio(q, cq)
            + 0.25 * fuzz.token_set_ratio(q, cq)
            + 0.20 * fuzz.WRatio(q, cq)
            + 0.20 * (100.0 * lev_sim(q, cq))
            + 0.10 * (
                100.0 * np.mean([
                    float(norm(query_row["topic"]) == norm(cand["topic"])),
                    float(norm(query_row["crop"]) == norm(cand["crop"])),
                    float(norm(query_row["agro_zone"]) == norm(cand["agro_zone"])),
                ])
            )
        )
        scores.append((score, i))

    return reference(local.iloc[max(scores)[1]]["reference_answer"])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

reports = []

for fold, (fit_idx, val_idx) in enumerate(kf.split(train), 1):
    fit_df = train.iloc[fit_idx].reset_index(drop=True)
    val_df = train.iloc[val_idx].reset_index(drop=True)

    model, feature_names = train_reranker(fit_df)

    for _, row in val_df.iterrows():
        pred, details = predict_with_reranker(row, fit_df, model)
        ref = reference(row["reference_answer"])

        base = lexical_baseline_predict(row, fit_df)

        reports.append({
            "fold": fold,
            "question": row["question"],
            "reference": ref,
            "reranker_prediction": pred,
            "reranker_distance": Levenshtein.distance(pred, ref),
            "baseline_prediction": base,
            "baseline_distance": Levenshtein.distance(base, ref),
            "candidate_count": details.get("candidate_count", 0),
            "predicted_distance": details.get("predicted_distance", np.nan),
            "routes": details.get("routes", []),
        })

oof = pd.DataFrame(reports)

print(f"Outer-fold reranker mean Levenshtein: {oof['reranker_distance'].mean():.3f}")
print(f"Outer-fold lexical baseline mean:     {oof['baseline_distance'].mean():.3f}")
print(f"Rows where reranker wins: {(oof['reranker_distance'] < oof['baseline_distance']).sum()} / {len(oof)}")
print(f"Rows where reranker ties: {(oof['reranker_distance'] == oof['baseline_distance']).sum()} / {len(oof)}")

print("\nWorst reranker cases:")
display(
    oof.sort_values("reranker_distance", ascending=False)[
        ["question", "reference", "reranker_prediction", "reranker_distance",
         "baseline_prediction", "baseline_distance"]
    ].head(12)
)


Outer-fold reranker mean Levenshtein: 49.822
Outer-fold lexical baseline mean:     51.711
Rows where reranker wins: 19 / 45
Rows where reranker ties: 18 / 45

Worst reranker cases:


,question,reference,reranker_prediction,reranker_distance,baseline_prediction,baseline_distance
14,Why are my maize lower leaves turning yellow b...,Likely nitrogen deficiency — yellowing starts ...,Uniform yellowing on older leaves with greener...,63,Uniform yellowing on older leaves with greener...,63
23,Yellow maize from the ground up — midrib still...,Uniform yellowing on older leaves with greener...,Likely nitrogen deficiency — yellowing starts ...,63,Likely nitrogen deficiency — yellowing starts ...,63
34,When should I treat fall armyworm?,When more than ten percent of plants are infes...,Larvae feed in the whorl leaving ragged window...,59,Larvae feed in the whorl leaving ragged window...,59
1,My soil test shows pH 5.2 — what should I do?,Apply agricultural lime based on the test and ...,Acidity binds phosphorus and limits nodulation...,59,Acidity binds phosphorus and limits nodulation...,59
11,Holes and sawdust in my maize whorl — what pest?,Fall armyworm whorl feeding; scout weekly and ...,Dry to twelve to thirteen percent and seal in ...,58,Stem borer tunneling kills the growing point i...,59
6,Sticky soot on seedlings in the nursery.,Aphid honeydew leading to sooty mould; wash ap...,Stem borer tunneling kills the growing point i...,58,"Monitor daily, use reflective mulch, and wash ...",63
37,Target moisture for cassava chips?,Below fourteen percent with uniform slices and...,They slow runoff and increase infiltration on ...,57,Dry chips below fourteen percent moisture befo...,52
38,Flies bothering cattle at the river evening.,Rotate grazing from riverine bush and use smok...,Follow extension eye-drop or drinking-water sc...,57,Smoke-treated pens reduce fly exposure when co...,56
29,How does night corralling help cattle in tsets...,Smoke-treated pens reduce fly exposure when co...,Rotate grazing from riverine bush and use smok...,56,Rotate grazing from riverine bush and use smok...,56
20,How do I know compost is ready to use?,"It should be dark, crumbly, and free of undeco...",Mucuna or lablab reduce erosion and suppress w...,56,"Provide shade, night grazing, and clean water;...",55


## 8. Feature importance and failure inspection

If the learned model is using nonsense signals, stop here rather than submitting it.

The feature importances are diagnostic only; they are not used as hand-written weights.


In [8]:

# Train one diagnostic model on all labelled rows.
diagnostic_model, feature_names = train_reranker(train)

importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": diagnostic_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
)

display(importance.head(15))


,feature,importance
17,answer_len,0.511428
18,question_len,0.184983
10,doc_char,0.052281
19,answer_question_len_ratio,0.045896
16,route_score,0.025557
4,q_token_sort,0.020799
5,q_wratio,0.019654
9,doc_word,0.014736
20,answer_q_word_overlap,0.011801
11,meta_topic,0.011649


## 9. Final training and test inference

Now that the algorithm has survived outer-fold validation, train the reranker on all 45 labelled rows.

The final submission still contains **only original training reference strings**. No generated text is silently inserted.


In [9]:

final_model, final_feature_names = train_reranker(train)

final_rows = []

for _, row in test.iterrows():
    answer, details = predict_with_reranker(
        row,
        train,
        final_model,
    )

    final_rows.append({
        "QuestionId": row["QuestionId"],
        "Answer": answer,
        "predicted_distance": details.get("predicted_distance", np.nan),
        "candidate_count": details.get("candidate_count", 0),
        "routes": str(details.get("routes", [])),
    })

final_debug = pd.DataFrame(final_rows)
submission = final_debug[["QuestionId", "Answer"]].copy()

submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
final_debug.to_csv(OUTPUT_DIR / "submission_debug.csv", index=False)

print("Wrote:", (OUTPUT_DIR / "submission.csv").resolve())
display(final_debug)


Wrote: /home/ui_hamilton/Documents/personal/agriculture-climate-slm-challenge/submission.csv


,QuestionId,Answer,predicted_distance,candidate_count,routes
0,1001,Follow extension eye-drop or drinking-water sc...,53.511576,19,"[(0.44921401742811307, 'doc_dis_001'), (0.3394..."
1,1002,Payouts may not match your individual field lo...,52.296006,19,"[(0.30767426017710947, 'doc_liv_002'), (0.2656..."
2,1003,Ensure balanced NPK including potassium for st...,53.663530,13,"[(0.37736559020845084, 'doc_dis_002'), (0.2649..."
3,1004,Follow extension eye-drop or drinking-water sc...,53.173935,17,"[(0.3565311949245591, 'doc_fer_002'), (0.29105..."
4,1005,Mucuna or lablab reduce erosion and suppress w...,50.434386,16,"[(0.39512208818259675, 'doc_fer_002'), (0.3063..."
5,1006,"Before hard seed set, leaving residues as surf...",51.783307,18,"[(0.3981966649605382, 'doc_pes_002'), (0.23876..."
6,1007,Acidity binds phosphorus and limits nodulation...,50.498571,16,"[(0.42682452666346704, 'doc_soi_001'), (0.2948..."
7,1008,Stem borer tunneling kills the growing point i...,44.749762,17,"[(0.4604981779754308, 'doc_pes_002'), (0.25760..."
8,1009,Payouts may not match your individual field lo...,50.933120,17,"[(0.4874022721855963, 'doc_soi_003'), (0.24131..."
9,1010,Use short-season drought-tolerant varieties al...,49.842822,16,"[(0.4412041244256796, 'doc_cli_001'), (0.31533..."


## 10. Submission validation

The CSV must preserve the test row order and contain exactly two columns.


In [10]:

sub_file = OUTPUT_DIR / "submission.csv"
assert sub_file.exists()

final_sub = pd.read_csv(sub_file)

assert list(final_sub.columns) == ["QuestionId", "Answer"]
assert len(final_sub) == len(test)
assert final_sub["QuestionId"].tolist() == test["QuestionId"].tolist()
assert final_sub["QuestionId"].is_unique
assert final_sub["Answer"].notna().all()
assert (final_sub["Answer"].astype(str).str.strip().str.len() > 0).all()

print(f"Submission validated: {len(final_sub)} rows")
display(final_sub)


Submission validated: 12 rows


,QuestionId,Answer
0,1001,Follow extension eye-drop or drinking-water sc...
1,1002,Payouts may not match your individual field lo...
2,1003,Ensure balanced NPK including potassium for st...
3,1004,Follow extension eye-drop or drinking-water sc...
4,1005,Mucuna or lablab reduce erosion and suppress w...
5,1006,"Before hard seed set, leaving residues as surf..."
6,1007,Acidity binds phosphorus and limits nodulation...
7,1008,Stem borer tunneling kills the growing point i...
8,1009,Payouts may not match your individual field lo...
9,1010,Use short-season drought-tolerant varieties al...


## 11. Optional local SLM

A local SLM can be added later as a **candidate generator/routing signal**, but it is intentionally not on the critical path.

If you already have a local Transformers model, the safe architecture is:

**SLM prediction → compare generated answer against known reference candidates → optionally add that similarity as another reranker feature.**

Do not point this notebook at a Hugging Face model ID. If the model is not already local, leave this section disabled.

The retrieval/reranking submission above remains reproducible without it.


In [11]:

USE_LOCAL_SLM = False
LOCAL_MODEL_PATH = ""

if USE_LOCAL_SLM:
    p = Path(LOCAL_MODEL_PATH).expanduser()
    if not p.is_dir() or not (p / "config.json").exists():
        raise ValueError(
            "LOCAL_MODEL_PATH must be an existing local Transformers model directory "
            "containing config.json."
        )
    print("Local SLM path accepted:", p)
else:
    print("Local SLM disabled; submission uses metric-aware reference reranking.")


Local SLM disabled; submission uses metric-aware reference reranking.


## 12. Experiment discipline

For each new version:

1. run this notebook locally;
2. record the outer-fold Levenshtein;
3. inspect the worst cases;
4. generate `submission.csv`;
5. upload it to Kaggle;
6. record the leaderboard score.

Do not change multiple components and then attribute a leaderboard movement to one change.

The important comparison is:

**v6 hybrid heuristic → v7 metric-aware reranker → optional local-SLM augmentation**

The objective is not a prettier pipeline. It is a lower measured Levenshtein distance.
